In [78]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import numpy as np
import plotly.figure_factory as ff
from shapely.geometry import Point
from plotly.subplots import make_subplots
import plotly.graph_objects as go
import json


In [79]:
import sys
!{sys.executable} -m pip install geopandas
import geopandas as gpd
#!{sys.executable} -m pip install --upgrade pip


In [80]:
fp = "tmpe9lqqyyv.csv"
df = pd.read_csv(fp)
print(len(df))
print(df.head())


/var/folders/pc/7dj9yy6d0yj9lhpgvnlq86wr0000gp/T/ipykernel_49176/618245389.py:2: DtypeWarning: Columns (23) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(fp)


878576
         businessname dbaname    legalowner         namelast  \
0  1000 Degrees Pizza     NaN  KHOSLA VIPAN  Pasquriello LLC   
1  1000 Degrees Pizza     NaN  KHOSLA VIPAN  Pasquriello LLC   
2  1000 Degrees Pizza     NaN  KHOSLA VIPAN  Pasquriello LLC   
3  1000 Degrees Pizza     NaN  KHOSLA VIPAN  Pasquriello LLC   
4  1000 Degrees Pizza     NaN  KHOSLA VIPAN  Pasquriello LLC   

              namefirst  licenseno                 issdttm  \
0  Kenneth Pasquariello     313440  2017-08-14 12:49:37+00   
1  Kenneth Pasquariello     313440  2017-08-14 12:49:37+00   
2  Kenneth Pasquariello     313440  2017-08-14 12:49:37+00   
3  Kenneth Pasquariello     313440  2017-08-14 12:49:37+00   
4  Kenneth Pasquariello     313440  2017-08-14 12:49:37+00   

                  expdttm licstatus licensecat  ...                violdttm  \
0  2020-01-01 04:59:00+00  Inactive         FS  ...  2018-03-20 14:54:25+00   
1  2020-01-01 04:59:00+00  Inactive         FS  ...  2018-03-20 14:54:25+00  

In [81]:
print(df['location'].isnull().sum()) #78765

missing_location = df[df['location'].isna()]
print(missing_location[['businessname', 'location', 'issdttm' ]])


79046
                businessname location                 issdttm
582     129 Lake Street Cafe      NaN  2013-10-24 12:48:40+00
583     129 Lake Street Cafe      NaN  2013-10-24 12:48:40+00
584     129 Lake Street Cafe      NaN  2013-10-24 12:48:40+00
585     129 Lake Street Cafe      NaN  2013-10-24 12:48:40+00
586     129 Lake Street Cafe      NaN  2013-10-24 12:48:40+00
...                      ...      ...                     ...
878421   Zumas Tex-Mex Grill      NaN  2012-02-22 15:46:22+00
878422   Zumas Tex-Mex Grill      NaN  2012-02-22 15:46:22+00
878423   Zumas Tex-Mex Grill      NaN  2012-02-22 15:46:22+00
878424   Zumas Tex-Mex Grill      NaN  2012-02-22 15:46:22+00
878425   Zumas Tex-Mex Grill      NaN  2012-02-22 15:46:22+00

[79046 rows x 3 columns]


In [82]:
#print(df.head())
print(df.iloc[1])
violation_counts = df["violdesc"].value_counts()
#print(violation_counts)
violation_levels = df["viol_level"].value_counts()
#print(violation_levels)


businessname                                   1000 Degrees Pizza
dbaname                                                       NaN
legalowner                                           KHOSLA VIPAN
namelast                                          Pasquriello LLC
namefirst                                    Kenneth Pasquariello
licenseno                                                  313440
issdttm                                    2017-08-14 12:49:37+00
expdttm                                    2020-01-01 04:59:00+00
licstatus                                                Inactive
licensecat                                                     FS
descript                                        Eating & Drinking
result                                                    HE_Fail
resultdttm                                 2018-03-20 14:54:25+00
violation                                         22-4-601/602.11
viol_level                                                     **
violdesc  

In [83]:
#Data Cleaning
df = df[df["licensecat"] == "FS"]
print(len(df)) #425589

#df = df.drop_duplicates()
df["violdttm"] = pd.to_datetime(df["violdttm"], errors = "coerce")
df["issdttm"] = pd.to_datetime(df["issdttm"], errors = "coerce")
df["expdttm"] = pd.to_datetime(df["expdttm"], errors = "coerce")

df["violdesc"] = df["violdesc"].str.strip()
df["businessname"] = df["businessname"].str.strip()


#Zip code cleaning
#df = df.dropna(subset=['zip'])
df['zip'] = df['zip'].astype(str)
df['zip'] = df['zip'].str.split('.').str[0]
df = df[df['zip'].str.isnumeric()]
df['zip'] = df['zip'].str.zfill(5)
print(df['zip'].unique())
print("Number of unique ZIP codes:", df['zip'].nunique())


428418
['02108' '02118' '02110' '02131' '02125' '02116' '02114' '02135' '02129'
 '02111' '02134' '02109' '02128' '02132' '02115' '02122' '02215' '02199'
 '02210' '02130' '02127' '02119' '02113' '02136' '02188' '02120' '02124'
 '02126' '02201' '02121' '02163' '02446' '02467']
Number of unique ZIP codes: 33


In [84]:
def extract_text_features(df):
    df['flag_hygiene'] = df['comments'].str.contains('hand|wash|glove|touch', case = False, na = False).astype(int)
    df['flag_temperature'] = df['comments'].str.contains('temp|cold|hot|thermometer', case = False, na = False).astype(int)
    df['flag_pests'] = df['comments'].str.contains('rodent|pest|dropping|fly', case = False, na = False).astype(int)
    df['flag_debris'] = df['comments'].str.contains('debris|grease|dirty|clean|unhygienic', case = False, na = False).astype(int)

    return df

df = extract_text_features(df)

In [85]:
dp03_df = pd.read_csv('ACSDP5Y2024.DP03-Data.csv')
print(dp03_df.columns)
print(len(dp03_df)) #236
dp03_df['fips_tract'] = dp03_df['GEO_ID'].str.split('US').str[1]
dp03_df = dp03_df.iloc[1:].copy()



Index(['GEO_ID', 'NAME', 'DP03_0001E', 'DP03_0001M', 'DP03_0002E',
       'DP03_0002M', 'DP03_0003E', 'DP03_0003M', 'DP03_0004E', 'DP03_0004M',
       ...
       'DP03_0133PM', 'DP03_0134PE', 'DP03_0134PM', 'DP03_0135PE',
       'DP03_0135PM', 'DP03_0136PE', 'DP03_0136PM', 'DP03_0137PE',
       'DP03_0137PM', 'Unnamed: 550'],
      dtype='object', length=551)
236


In [86]:
print(len(df)) #425327
print(df['location'].isnull().sum()) #32131
#7% of missing data. 
#print(df.head())
print(df.iloc[1])

428156
32309
businessname                                       1000 Degrees Pizza
dbaname                                                           NaN
legalowner                                               KHOSLA VIPAN
namelast                                              Pasquriello LLC
namefirst                                        Kenneth Pasquariello
licenseno                                                      313440
issdttm                                     2017-08-14 12:49:37+00:00
expdttm                                     2020-01-01 04:59:00+00:00
licstatus                                                    Inactive
licensecat                                                         FS
descript                                            Eating & Drinking
result                                                        HE_Fail
resultdttm                                     2018-03-20 14:54:25+00
violation                                             22-4-601/602.11
viol_le

In [87]:
#Extract census tracts
import ast

def parse_location(val):
    if isinstance(val, tuple):
        return val
    if isinstance(val, str):
        return ast.literal_eval(val)
    return None

df['location'] = df['location'].apply(parse_location)
df = df.dropna(subset = ['location'])



df['latitude'] = df['location'].apply(lambda x: x[0])
df['longitude'] = df['location'].apply(lambda x: x[1])




df['geometry'] = gpd.points_from_xy(df['longitude'], df['latitude'])

gdf = gpd.GeoDataFrame(df, geometry = 'geometry', crs = 'EPSG:4326')




In [88]:
#Shapefiles for each period 

tract_2000 = gpd.read_file('Shapefiles/tl_2010_25_tract00.shp')
tract_2010 = gpd.read_file('Shapefiles/tl_2010_25_tract10.shp')
tract_2020 = gpd.read_file('Shapefiles/tl_2020_25_tract.shp')
print(tract_2010.columns)

tract_2000 = tract_2000.set_crs(epsg = 4269).to_crs(epsg = 4326)
tract_2010 = tract_2010.set_crs(epsg = 4269).to_crs(epsg = 4326)
tract_2020 = tract_2020.set_crs(epsg = 4269).to_crs(epsg = 4326)
print(tract_2020.columns)

Index(['STATEFP10', 'COUNTYFP10', 'TRACTCE10', 'GEOID10', 'NAME10',
       'NAMELSAD10', 'MTFCC10', 'FUNCSTAT10', 'ALAND10', 'AWATER10',
       'INTPTLAT10', 'INTPTLON10', 'geometry'],
      dtype='object')
Index(['STATEFP', 'COUNTYFP', 'TRACTCE', 'GEOID', 'NAME', 'NAMELSAD', 'MTFCC',
       'FUNCSTAT', 'ALAND', 'AWATER', 'INTPTLAT', 'INTPTLON', 'geometry'],
      dtype='object')


In [89]:
gdf['year'] = pd.to_datetime(gdf['issdttm']).dt.year

gdf_2000 = gdf[(gdf['year'] >= 2006) & (gdf['year'] <= 2009)].copy()

gdf_2010 = gdf[(gdf['year'] >=2010) & (gdf['year'] <= 2019)].copy()

gdf_2020 = gdf[(gdf['year'] >=2020) & (gdf['year'] <=2026)].copy()

In [90]:
join_2000 = gpd.sjoin(gdf_2000, tract_2000, how = 'left', predicate = 'within')
join_2010 = gpd.sjoin(gdf_2010, tract_2010, how = 'left', predicate = 'within')
join_2020 = gpd.sjoin(gdf_2020, tract_2020, how='left', predicate='within')

gdf_final = pd.concat([join_2000, join_2010, join_2020], ignore_index = True)

#Reconcile Census ID Columns

id_cols = ['GEOID', 'GEOID10', 'CTIDFP00', 'STFID']
gdf_final['census_tract'] = None

for col in id_cols:
    if col in gdf_final.columns:
        gdf_final['census_tract'] = gdf_final['census_tract'].fillna(gdf_final[col])

#Clean-up
cols_to_drop = ['index_right'] + [c for c in id_cols if c in gdf_final.columns]
df_final = gdf_final.drop(columns = cols_to_drop, errors='ignore')

df_final['census_tract'] = df_final['census_tract'].astype(str).str.replace(r'\.0', '',regex=True).str.zfill(11) #import regex



In [91]:

#df_final.to_csv('BosInspector_withCensusTracts.csv', index=False)

gdf_final = gpd.GeoDataFrame(df_final, geometry='geometry', crs="EPSG:4326")
gdf_final.to_file('BosInspector_WithCensusTracts.geojson', driver='GeoJSON')
print(f"Success!, Procecssed {len(df_final)} records.")
print(df_final[['businessname', 'year', 'census_tract']].head())


Success!, Procecssed 395262 records.
    businessname    year census_tract
0     711 Bistro  2009.0  25025010700
1     711 Bistro  2009.0  25025010700
2        75 CAFE  2009.0  25025070200
3        75 CAFE  2009.0  25025070200
4  77 Restaurant  2009.0  25025091600


In [92]:
df_final['fips_tract'] = df_final['census_tract']

cols_to_keep = ['fips_tract', 'DP03_0062E', 'DP03_0128PE']
dp03_subset = dp03_df[cols_to_keep].copy()
dp03_subset['fips_tract'] = dp03_subset['fips_tract'].astype(str).str.replace(r'\.0', '', regex = True).str.zfill(11)
dp03_subset = dp03_subset.rename(columns = {
    'DP03_0062E': 'median_income',
    'DP03_0128PE': 'pct_poverty'
})

dp03_subset['median_income'] = pd.to_numeric(dp03_subset['median_income'], errors = 'coerce')
dp03_subset['pct_poverty'] = pd.to_numeric(dp03_subset['pct_poverty'], errors = 'coerce')
print(dp03_subset.head())
#print(dp03_subset.head())
analysis_df = pd.merge(
    df_final,
    dp03_subset,
    on = 'fips_tract',
    how = 'inner'
)


print(analysis_df.head())
print(len(analysis_df))


    fips_tract  median_income  pct_poverty
1  25025000101       165469.0         11.3
2  25025000102        86875.0         21.7
3  25025000201       123525.0         17.1
4  25025000202        72098.0         25.9
5  25025000301       132900.0         11.4
    businessname dbaname                legalowner                  namelast  \
0  77 Restaurant     NaN               NGUYEN LINH                      Dang   
1  77 Restaurant     NaN               NGUYEN LINH                      Dang   
2  77 Restaurant     NaN               NGUYEN LINH                      Dang   
3    ACES MARKET     NaN  OTIS STEELE & DAVE DARLI  OTIS STEELE & DAVE DARLI   
4    ACES MARKET     NaN  OTIS STEELE & DAVE DARLI  OTIS STEELE & DAVE DARLI   

  namefirst  licenseno                   issdttm                   expdttm  \
0   Charlie      31333 2009-05-14 19:08:23+00:00 2010-01-01 04:59:00+00:00   
1   Charlie      31333 2009-05-14 19:08:23+00:00 2010-01-01 04:59:00+00:00   
2   Charlie      31333 2009

In [104]:
cols_to_keep = [
    'businessname', "licenseno", 'descript', 'address', 'zip', 

    'latitude', 'longitude', 'census_tract',

    'result', 'viol_level', 'viol_status', 'violdesc', 'comments',

    'year', 'issdttm', 'resultdttm',

    'fips_tract','median_income', 'pct_poverty',
    
    'flag_hygiene', 'flag_temperature', 'flag_pests', "flag_debris"
]

analysis_df = analysis_df[cols_to_keep].copy()

analysis_df['resultdttm'] = pd.to_datetime(analysis_df['resultdttm'], format = 'ISO8601')
analysis_df = analysis_df.sort_values(['licenseno', 'resultdttm'])
analysis_df['inspection_gap'] = analysis_df.groupby('licenseno')['resultdttm'].diff().dt.days

analysis_df['inspection_gap'] = analysis_df['inspection_gap'].fillna(analysis_df['inspection_gap'].median())

print(analysis_df.head())

                       businessname  licenseno           descript  \
133111  Morning Star Baptist Church      17613  Eating & Drinking   
133112  Morning Star Baptist Church      17613  Eating & Drinking   
133113  Morning Star Baptist Church      17613  Eating & Drinking   
133114  Morning Star Baptist Church      17613  Eating & Drinking   
133115  Morning Star Baptist Church      17613  Eating & Drinking   

                     address    zip  latitude  longitude census_tract  \
133111  1257    BLUE HILL AV  02126  42.28176  -71.09266  25025101101   
133112  1257    BLUE HILL AV  02126  42.28176  -71.09266  25025101101   
133113  1257    BLUE HILL AV  02126  42.28176  -71.09266  25025101101   
133114  1257    BLUE HILL AV  02126  42.28176  -71.09266  25025101101   
133115  1257    BLUE HILL AV  02126  42.28176  -71.09266  25025101101   

         result viol_level  ...                   issdttm  \
133111  HE_Fail          *  ... 2012-01-18 21:49:03+00:00   
133112  HE_Fail         

In [105]:
#Further data cleaning for final analysis dataset

analysis_df['is_fail'] = (analysis_df['viol_status'] == 'Fail').astype(int)

analysis_df['viol_score'] = analysis_df["viol_level"].str.count(r'\*').fillna(0)

aggregations = {
    'businessname': 'first', 
    'address': 'first', 
    'fips_tract': 'first', 
    'median_income': 'first',
    'pct_poverty': 'first',
    'latitude': 'first',
    'longitude': 'first',
    'is_fail': 'sum', 
    'viol_score': 'sum',
    'result': 'count',
    'flag_hygiene': 'max', 
    'flag_temperature': 'max', 
    'flag_pests': 'max',
    'flag_debris': 'max', 
    'inspection_gap': 'mean'

}

final_analysis_df = analysis_df.groupby(['licenseno', 'year']).agg(aggregations).reset_index()
final_analysis_df = final_analysis_df.rename(columns = {
    'result': 'inspection_count',
    'is_fail': 'total_failures'
})

final_analysis_df['failure_rate'] = final_analysis_df['total_failures']/final_analysis_df['inspection_count']

print(final_analysis_df.head())
print(final_analysis_df.columns)


   licenseno    year                 businessname               address  \
0      17613  2012.0  Morning Star Baptist Church  1257    BLUE HILL AV   
1      18013  2011.0                  Twelve Bens         315  ADAMS ST   
2      18015  2011.0           Sonny's Restaurant         750  ADAMS ST   
3      18022  2012.0       GERARD'S SANDWICH SHOP           772   ADAMS   
4      18026  2011.0                     Erie Pub         791  ADAMS ST   

    fips_tract  median_income  pct_poverty   latitude  longitude  \
0  25025101101        93750.0         17.2  42.281760 -71.092660   
1  25025092101        66210.0         11.5  42.298605 -71.057578   
2  25025100700       121122.0          4.8  42.283970 -71.055271   
3  25025100700       121122.0          4.8  42.283450 -71.055511   
4  25025100800        78533.0         12.1  42.283160 -71.056020   

   total_failures  viol_score  inspection_count  flag_hygiene  \
0              19        44.0                64             1   
1         

In [ ]:
#import sys
#!{sys.executable} -m pip install statsmodels

  Using cached statsmodels-0.14.6-cp313-cp313-macosx_11_0_arm64.whl.metadata (9.5 kB)
  Using cached scipy-1.17.1-cp313-cp313-macosx_14_0_arm64.whl.metadata (62 kB)
  Using cached patsy-1.0.2-py2.py3-none-any.whl.metadata (3.6 kB)
Using cached statsmodels-0.14.6-cp313-cp313-macosx_11_0_arm64.whl (10.0 MB)
Using cached patsy-1.0.2-py2.py3-none-any.whl (233 kB)
Using cached scipy-1.17.1-cp313-cp313-macosx_14_0_arm64.whl (20.3 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [statsmodels] [statsmodels]


In [107]:
#Neighborhood significance

import statsmodels.api as sm

df_model = final_analysis_df.dropna(subset=['median_income', 'pct_poverty', 'failure_rate', 'flag_hygiene'])

X_base = df_model[['inspection_count', 'inspection_gap', 'flag_hygiene', 'flag_pests', 'flag_debris']]
X_base = sm.add_constant(X_base)

model_base = sm.OLS(df_model['failure_rate'], X_base).fit()

#Full model 
X_nb = df_model[['inspection_count', 'inspection_gap', 'flag_hygiene', 'flag_pests', 'flag_debris', 'median_income', 'pct_poverty']]
X_nb = sm.add_constant(X_nb)
model_nb = sm.OLS(df_model['failure_rate'], X_nb).fit()

#Compare the two models

print(f"Base model R-squared: {model_base.rsquared:.4f}| AIC: {model_base.aic:.2f}")
print(f"Full model R-squared: {model_nb.rsquared:.4f}| AIC: {model_nb.aic:.2f}")

#Conclusion: Neighborhoods do matter. Adding Census data makes the model better at explaining Boston's food safety landscape.

Base model R-squared: 0.6121| AIC: -4115.00
Full model R-squared: 0.6145| AIC: -4125.39


In [108]:
X_final_cols = ['inspection_count', 'inspection_gap', 'flag_hygiene', 'flag_pests', 'flag_debris', 'median_income', 'pct_poverty']
X_final = sm.add_constant(df_model[X_final_cols])
model_final = sm.OLS(df_model['failure_rate'], X_final).fit()


print(model_final.summary())

                            OLS Regression Results                            
Dep. Variable:           failure_rate   R-squared:                       0.614
Model:                            OLS   Adj. R-squared:                  0.613
Method:                 Least Squares   F-statistic:                     534.7
Date:                Mon, 06 Apr 2026   Prob (F-statistic):               0.00
Time:                        21:15:17   Log-Likelihood:                 2070.7
No. Observations:                2356   AIC:                            -4125.
Df Residuals:                    2348   BIC:                            -4079.
Df Model:                           7                                         
Covariance Type:            nonrobust                                         
                       coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------------
const                0.2075      0.012  

In [ ]:
tract_df = final_analysis_df.groupby('fips_tract').agg({
    'failure_rate': 'mean', 
    'median_income': 'first',
    'businessname': 'count'
}).reset_index().rename(columns = {'businessname': 'business_count'})

with open('BosInspector_WithCensusTracts.json') as f:
    geojson = json.load(f)



In [ ]:
geo_df = final_analysis_df.dropna(subset=['latitude', 'longitude', 'failure_rate'])

fig = px.density_map(
    geo_df, 
    lat='latitude', 
    lon='longitude', 
    z='failure_rate',        
    radius=15,               
    center=dict(lat=42.3601, lon=-71.0589), # Center on Boston
    zoom=11,
    map_style="carto-positron", 
    title="Geographical Heatmap: Hotspots of Inspection Failures",
    color_continuous_scale="OrRd" 
)

fig.show()

In [20]:
fig.write_html('YehS.graphHomework10.html')

In [16]:
df_fail = df[df["viol_status"] == "Fail"]
main_levels = ["*", "**", "***"]
df_fail = df_fail[df_fail["viol_level"].isin(main_levels)]

df_fail['year'] = df_fail['violdttm'].dt.year
print(df_fail['zip'].unique())
viol_business_count = df_fail.groupby(['businessname','year','viol_level']).size().reset_index(name='num_violations')
print(viol_business_count.head(10))
viol_zip_count = df_fail.groupby(['zip','year','viol_level']).size().reset_index(name='num_violations')
print(viol_zip_count.head(10))


['02108' '02118' '02131' '02125' '02116' '02114' '02135' '02129' '02111'
 '02110' '02134' '02109' '02128' '02132' '02115' '02122' '02215' '02199'
 '02210' '02130' '02127' '02119' '02113' '02136' '02188' '02120' '02124'
 '02126' '02201' '02121' '02163' '02446' '02467']
               businessname  year viol_level  num_violations
0  100 Percent Delicia Food  2013          *              16
1  100 Percent Delicia Food  2013        ***               1
2  100 Percent Delicia Food  2014          *               3
3  100 Percent Delicia Food  2015          *              13
4  100 Percent Delicia Food  2015        ***               1
5  100 Percent Delicia Food  2016          *              18
6  100 Percent Delicia Food  2016        ***               5
7  100 Percent Delicia Food  2017          *              19
8  100 Percent Delicia Food  2017        ***               3
9  100 Percent Delicia Food  2018          *              11
     zip  year viol_level  num_violations
0  02108  2007    

In [ ]:
#Prepare data for location
df_fail = df_fail[df_fail['location'].notnull()]

def extract_latlon(x):
    try: 
        if isinstance(x, (tuple, list)):
            return float(x[0]), float(x[1])
        elif isinstance(x, str):
            parts = x.replace('(', '').replace(')', '').split(',')
            return float(parts[0]), float(parts[1])
    except:
        return np.nan, np.nan

df_fail['lat'], df_fail['lon'] = zip(*df_fail['location'].apply(extract_latlon))


df_fail = df_fail[df_fail['lat'].notnull() & df_fail['lon'].notnull()]

# Ensure types are numeric
df_fail['lat'] = df_fail['lat'].astype(float)
df_fail['lon'] = df_fail['lon'].astype(float)


In [30]:
# Round coordinates to ~0.01 degrees (~1 km) to define clusters
df_fail['lat_bin'] = df_fail['lat'].round(2)   # replace 'latitude' with your column name
df_fail['lon_bin'] = df_fail['lon'].round(2)  # replace 'longitude' with your column name

# Combine rounded coordinates to define a cluster
df_fail['cluster'] = df_fail['lat_bin'].astype(str) + "_" + df_fail['lon_bin'].astype(str)

print(df_fail[['businessname', 'cluster']].drop_duplicates().head(10))


                                        businessname       cluster
0                                 1000 Degrees Pizza  42.36_-71.06
13                              1000 Washington Cafe  42.35_-71.06
48                          100 Percent Delicia Food  42.28_-71.12
316                                        110 Grill  42.33_-71.06
533                              11 Dining -16th Fl.  42.35_-71.07
564                        125 Nashua St. Cafe (MGH)  42.37_-71.06
744   150 Boylston St. Dining Room @ Emerson College  42.35_-71.07
877                          163 Vietnamese Sandwich  42.35_-71.06
1120                                1928 Beacon Hill  42.36_-71.07
1168                                1928 Rowes Wharf  42.36_-71.05


In [58]:
viol_cluster_count = (
    df_fail.groupby(['cluster', 'year', 'viol_level'])
    .size()
    .reset_index(name = 'num_violations')
)

print(viol_cluster_count)



           cluster  year viol_level  num_violations
0     42.24_-71.13  2008          *              10
1     42.24_-71.13  2008         **               1
2     42.24_-71.13  2008        ***               7
3     42.24_-71.13  2009          *              18
4     42.24_-71.13  2009         **               2
...            ...   ...        ...             ...
5794    42.4_-71.0  2019         **               4
5795    42.4_-71.0  2019        ***               1
5796    42.4_-71.0  2022         **               2
5797    42.4_-71.0  2022        ***               1
5798    42.4_-71.0  2024          *               2

[5799 rows x 4 columns]


In [40]:
# Compute cluster coordinates and total violations
cluster_summary = (
    df_fail.groupby(['cluster', 'viol_level'])
    .agg(
        num_violations=('businessname', 'count'),  # count failed restaurants
        lat_avg=('lat', 'mean'),             # replace 'latitude' with your column name
        lon_avg=('lon', 'mean')             # replace 'longitude' with your column name
    )
    .reset_index()
)

# Optionally, compute total violations per cluster for sizing
cluster_total = cluster_summary.groupby('cluster').agg(
    total_violations=('num_violations', 'sum')
).reset_index()

cluster_summary = cluster_summary.merge(cluster_total, on='cluster')

In [60]:
severity_map = {"*": 1, "**": 2, "***": 3}
df_fail['severity_score'] = df_fail['viol_level'].map(severity_map)

In [70]:
# Round lat/lon to create small grid cells (~100m)
df_fail['lat_bin'] = df_fail['lat'].round(3)  # 0.001 deg ~ 100m
df_fail['lon_bin'] = df_fail['lon'].round(3)

# Aggregate violations per grid cell and severity
grid_summary = (
    df_fail.groupby(['lat_bin', 'lon_bin', 'viol_level'])
    .agg(total_severity = ('severity_score', 'sum'),
         businesses =("businessname", lambda x: ", ".join(sorted(set(x)))),
         num_violations = ("businessname", 'count')
    )
    .reset_index()
)
print(grid_summary)
# For a general heat, sum over all violation levels
grid_summary_total = (
    df_fail.groupby(['lat_bin', 'lon_bin'])
    .size()
    .reset_index(name='num_violations')
)

      lat_bin  lon_bin viol_level  total_severity  \
0      42.238  -71.132          *              62   
1      42.238  -71.132         **              10   
2      42.238  -71.132        ***              48   
3      42.242  -71.127          *               4   
4      42.242  -71.127         **              14   
...       ...      ...        ...             ...   
3238   42.394  -71.000         **              42   
3239   42.394  -71.000        ***              27   
3240   42.395  -71.000          *               6   
3241   42.395  -71.000         **              12   
3242   42.395  -71.000        ***              15   

                                             businesses  num_violations  
0                                 Readville Tavern Inc.              62  
1                                 Readville Tavern Inc.               5  
2                                 Readville Tavern Inc.              16  
3                                   Stop & Shop No. 413            

In [71]:

cluster_summary['viol_level'] = cluster_summary['viol_level'].astype(str)
fig = px.density_mapbox(
    grid_summary,                   
    lat='lat_bin',
    lon='lon_bin',
    z='total_severity',                   
    radius=15,
    hover_name = 'businesses',
    hover_data = {
        'total_severity': True,
        'num_violations': True,
        'lat_bin': False,
        'lon_bin': False
    },                             
    center=dict(lat=df_fail['lat'].mean(), lon=df_fail['lon'].mean()),  
    zoom=12,
    mapbox_style="carto-positron",
    title="Geospatial Heatmap of Failed Restaurant Violations"
)

fig.show()

/var/folders/pc/7dj9yy6d0yj9lhpgvnlq86wr0000gp/T/ipykernel_61269/1569978487.py:2: DeprecationWarning:

*density_mapbox* is deprecated! Use *density_map* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/



In [ ]:
fig = px.line(
    viol_zip_count, 
    x = 'year', 
    y = 'num_violations', 
    color = 'viol_level',
    line_group = 'zip',
    hover_name = 'zip',
    markers = True,
    title = 'Timeline of Violations by ZIP Code (2006-2026)'
)
trace = (
    viol_zip_count[['zip', 'viol_level']]
    .drop_duplicates()
    .sort_values(['viol_level', 'zip'])
)

zips = sorted(viol_zip_count['zip'].unique())
zip_buttons = []
zip_buttons.append(dict(
    label = "All ZIPs",
    method = 'update',
    args =[
        {'visible': [True] * len(fig.data)}, 
        {'title': "Violations Over Time for All ZIPs"}
    ]


))

for z in zips:
    zip_buttons.append(dict(
        label = str(z),
        method = "update",
        args = [
            {'visible': [row.zip == str(z) for row in trace.itertuples()]},
            {"title": f"Violations Over Time for ZIP {z}"}
        ]
    ))

fig.update_layout(
    updatemenus = [
        dict(
            buttons = [
                dict(
                    label = "All ZIPs",
                    method = 'update',
                    args = [{"visible": [True]*len(fig.data)},
                            {"title": "Violations Over Time for All ZIPs"}]
                )
            ] + zip_buttons,
            direction = 'down',
            showactive = True
        )
    ]
)

fig.update_layout(
    xaxis = dict(dtick = 1),
    yaxis_title = "Number of Violations",
    legend_title = "Violation Level",
    hovermode = 'x unified',
    title_x = 0.5,
    legend = dict(
        title = 'Violation Level',
        orientation = 'v',
        x = 1.02,
        y = 1
    )
)
fig.show()

In [72]:
fig.write_html('YehS.graphHomework8.html')